In [2]:
import os
import cv2
import torch
import numpy as np
from tqdm import tqdm

from vggt.models.vggt import VGGT
from vggt.utils.load_fn import load_and_preprocess_images
from vggt.utils.pose_enc import pose_encoding_to_extri_intri

/home/gokul/ideaslab/vggt_for_camera_params/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

cam1 = "calibration_images/cam1"
cam2 = "calibration_images/cam2"        
cam3 = "calibration_images/cam3"

output = "calibration_images/camera_params"

In [4]:
model = VGGT.from_pretrained("facebook/VGGT-1B").to(DEVICE)
model.eval()

VGGT(
  (aggregator): Aggregator(
    (patch_embed): DinoVisionTransformer(
      (patch_embed): PatchEmbed(
        (proj): Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14))
        (norm): Identity()
      )
      (blocks): ModuleList(
        (0-23): 24 x NestedTensorBlock(
          (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True, bias=True)
          (attn): MemEffAttention(
            (qkv): Linear(in_features=1024, out_features=3072, bias=True)
            (q_norm): Identity()
            (k_norm): Identity()
            (attn_drop): Dropout(p=0.0, inplace=False)
            (proj): Linear(in_features=1024, out_features=1024, bias=True)
            (proj_drop): Dropout(p=0.0, inplace=False)
          )
          (ls1): LayerScale()
          (drop_path1): Identity()
          (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True, bias=True)
          (mlp): Mlp(
            (fc1): Linear(in_features=1024, out_features=4096, bias=True)
            (a

In [5]:
cam1_files = sorted(os.listdir(cam1))
cam2_files = sorted(os.listdir(cam2))
cam3_files = sorted(os.listdir(cam3))

common_files = list(set(cam1_files) & set(cam2_files) & set(cam3_files))
common_files = common_files[:100]

In [6]:
img = cv2.imread("calibration_images/cam1/frame_0000.jpg")

H, W = img.shape[:2]

In [7]:
all_intrinsics = []
all_extrinsics = []

with torch.no_grad():
    for fname in tqdm(common_files):
        paths = [
            os.path.join(cam1, fname),
            os.path.join(cam2, fname),
            os.path.join(cam3, fname),
        ]

        images = load_and_preprocess_images(paths).to(DEVICE)

        outputs = model(images)

        pose_enc = outputs["pose_enc"]

        extrinsics, intrinsics = pose_encoding_to_extri_intri(pose_enc, image_size_hw=(H, W))

        all_intrinsics.append(intrinsics[0].cpu().numpy())
        all_extrinsics.append(extrinsics[0].cpu().numpy())

  0%|          | 0/66 [00:00<?, ?it/s]/home/gokul/ideaslab/vggt_for_camera_params/vggt/models/vggt.py:65: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):
100%|██████████| 66/66 [00:39<00:00,  1.66it/s]


In [8]:
all_extrinsics = np.array(all_extrinsics)
all_intrinsics = np.array(all_intrinsics)

np.save(os.path.join(output, "extrinsics.npy"), all_extrinsics)
np.save(os.path.join(output, "intrinsics.npy"), all_intrinsics)

print(f" Extrisnics shape: {all_extrinsics.shape} \n Intrinsics shape: {all_intrinsics.shape}")

 Extrisnics shape: (66, 3, 3, 4) 
 Intrinsics shape: (66, 3, 3, 3)
